# Generate-only eval pass — one PTO_Exp3 model state

Runs **just Step 1** of an iteration: load `iteration_N/adapter`, simulate the eval
conversations, write them to `model_iter_N_TT*_TP*/`. **No branching, no look-ahead, no
oracle, no DPO** — so it costs patient API calls + GPU generation only.

### Why this exists
`model_iter_k` is normally produced as Step 1 of iteration `k+1`. If a run dies between
"iteration k's adapter saved" and "iteration k+1's Step 1 finished", the adapter exists but
has **no eval data** — so it can never be scored. That is the PTO LA5 state: adapters for
iterations 1–5, but `model_iter_5_TT0.9_TP0.7/` is empty because iteration 6 was killed ~1
minute in.

### Fidelity
The config is rebuilt from the run's own `run_metadata.json` (which is `asdict(PTOConfig)`),
so this pass **cannot drift** from how iterations 0..N-1 were generated. The two seeds are
derived, not typed:

> `model_iter_k` ⇐ iteration `k+1`'s Step 1 ⇒ shuffle seed = patient seed = `cfg.seed + k + 1`

which is the same formula `eda_analysis.data.persona_order` replays to recover `persona_id`.
Section 3 **proves** it on this run before anything is spent.

### Host
**Run this on Colab.** Full-scale GPU work on the local Blackwell (RTX 5070 Ti, sm_120)
reboots the machine — including *inference-only* generation with no backward pass, confirmed
2026-07-30. Local is for `_local_smoke.py` / quicktest scale / EDA only.

Helpers live in `generate_eval_convs.py` (same folder), which is also runnable as a CLI:
`python generate_eval_convs.py --iter 5 --verify-seeds`. Generation is **resume-safe per
conversation CSV** — if the runtime drops, just re-run; finished conversations are skipped.

---
## 1. Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Which model state to evaluate. ITER = the MODEL STATE: loads iteration_{ITER}/adapter
# and writes model_iter_{ITER}/.  (Per the iter<->model-state mapping: the adapter from
# training iteration N generates the convs saved as model_iter_N.)
# ─────────────────────────────────────────────────────────────────────────────
ITER       = 5
EXPERIMENT = "PTO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_M8_PTgreedy"
MODE_TAG   = "full"          # "full" | "quicktest"

# ─────────────────────────────────────────────────────────────────────────────
# BATCH_SIZE — a VRAM lever, NOT a science knob (does not change sampled outputs).
#   None = the run's own value (96). Correct on an A100-80GB.
#
# ON THE 12 GB LOCAL CARD, MEASURED 2026-07-30 (weights alone = 2.6 GB):
#     batch 4  ->  7.1 GB peak   (58%)
#     batch 6  ->  8.0 GB peak   (65%)  <- per batch, WITH the inter-batch cache fix
#     batch 32 -> ~38 GB         (3.2x over -> REBOOTED the machine)
# An over-budget request on this GPU reboots the PC instead of raising OutOfMemoryError,
# so DO the ~1.1 GB/conversation arithmetic before raising this. 96 convs at batch 6 is
# ~16 batches x ~190 s ~= 50 min.
#
# Watch the `vram X.XG` field printed at the end of each batch line: it is
# torch.cuda.memory_reserved() and should stay FLAT across batches. If it climbs every
# batch, the inter-batch cache release in _shared/convs.py is missing/regressed — stop.
# ─────────────────────────────────────────────────────────────────────────────
BATCH_SIZE = 6              # None on Colab/A100; 4-6 on the 12 GB local card

# Prove the seed convention against the already-generated iterations before spending.
# Free (reads local CSVs only, ~30 s). Leave True.
VERIFY_SEEDS = True

print(f"model_iter_{ITER}  <-  iteration_{ITER}/adapter")
print(f"{EXPERIMENT} [{MODE_TAG}]")

---
## 2. Colab setup, imports, config

In [ ]:
# Colab pre-bakes torchao < 0.16.0, which peft 0.19.1 REJECTS by *raising* ImportError
# inside its LoRA dispatcher (dispatch_torchao). That dispatcher runs for
# PeftModel.from_pretrained too — not just get_peft_model — so THIS pass hits it as well,
# even though it never trains. The project never uses torchao, so removing it makes the
# dispatcher short-circuit. No-op off Colab (the local env is already torchao-free).
%pip uninstall -y -q torchao

# Uncomment on a FRESH Colab runtime to pin the validated dependency set (requirements.txt).
# torch is intentionally not pinned — Colab ships a CUDA build.
# %pip install -q accelerate==1.13.0 bitsandbytes==0.49.2 datasets==4.8.5 \
#              huggingface_hub==1.14.0 numpy==2.4.4 openai==2.36.0 pandas==3.0.3 \
#              peft==0.19.1 scipy==1.17.1 sentence-transformers==5.5.0 \
#              tensorboard==2.20.0 transformers==5.8.1 trl==1.4.0 wandb==0.26.1

In [ ]:
# ─── Bootstrap: mount Drive + cd into PTO_Exp3/, then put code/ on sys.path ───
# Mirrors train_PTO_Iterative.ipynb's bootstrap so imports resolve identically.
import os, sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/code/PTO_Exp3")

# Locate code/PTO_Exp3/ WITHOUT assuming cwd: a notebook has no __file__, and VS Code's
# working directory depends on the jupyter.notebookFileRoot setting (workspace root vs
# notebook dir). Probe cwd, then walk up looking for the marker file.
_here = None
for _cand in [os.getcwd()] + [os.path.abspath(os.path.join(os.getcwd(), *[".."] * _i))
                              for _i in range(1, 6)]:
    if os.path.isfile(os.path.join(_cand, "pto_trainer.py")):
        _here = _cand
        break
    _nested = os.path.join(_cand, "Exp3_PTO_GRPO", "code", "PTO_Exp3")
    if os.path.isfile(os.path.join(_nested, "pto_trainer.py")):
        _here = _nested
        break
if _here is None:
    raise RuntimeError(
        f"Could not locate code/PTO_Exp3/ (no pto_trainer.py) from cwd={os.getcwd()!r}. "
        "Open this notebook from inside Exp3_PTO_GRPO/code/PTO_Exp3/, or os.chdir() there."
    )

_code_dir = os.path.abspath(os.path.join(_here, ".."))
for _p in (_code_dir, _here):
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.chdir(_here)   # so detect_runtime's walk-up finds the key files at Exp3_PTO_GRPO/
print(f"PTO_Exp3: {_here}")
print(f"code/ on sys.path: {_code_dir}")

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
# generate_eval_convs FIRST: it imports trl before torch. On the local Blackwell (sm_120)
# `from trl import ...` AFTER torch segfaults at CUDA init (exit 139). Harmless on Colab,
# but keep the order so this notebook stays runnable in both places.
from generate_eval_convs import load_cfg, seeds_for, verify_seeds

import random
import torch
from peft import PeftModel

from _shared import (
    detect_runtime, init_openai_client, authenticate,
    setup_tokenizer, load_base_model, sync_pad_token, patch_generate,
    setup_permutations,
)
from pto_trainer import run_generation_only

rt = detect_runtime(run_env="auto", experiment_name="Exp3_PTO_GRPO")

In [ ]:
# ─── Config, rebuilt from the run's OWN run_metadata.json (= asdict(PTOConfig)) ───
# Colab-absolute local_outdir/conv_outdir are remapped onto this host, and the recomputed
# tail is asserted against the stored one so a rename can never silently misroute output.
cfg = load_cfg(rt.experiment_root, EXPERIMENT, MODE_TAG, BATCH_SIZE)

adapter_dir = os.path.join(cfg.local_outdir, f"iteration_{ITER}", "adapter")
conv_dir = os.path.join(
    cfg.conv_outdir,
    f"model_iter_{ITER}_TT{cfg.temperature_therapist_gen}_TP{cfg.temperature_patient}",
)
seed = seeds_for(cfg, ITER)   # cfg.seed + ITER + 1

assert os.path.isdir(adapter_dir), (
    f"No adapter at {adapter_dir} — iteration {ITER} never finished training."
)
existing = (sorted(f for f in os.listdir(conv_dir) if f.startswith("conversation_"))
            if os.path.isdir(conv_dir) else [])

print("=" * 70)
print(f"GENERATE-ONLY EVAL PASS — model_iter_{ITER}")
print("=" * 70)
print(f"  Adapter:      {adapter_dir}")
print(f"  Output:       {conv_dir}")
print(f"  Seeds:        shuffle = patient_api = {cfg.seed} + {ITER} + 1 = {seed}")
print(f"  Convs:        {cfg.num_conversations_per_iter} x {cfg.num_utterances_for_data} utts "
      f"(TT {cfg.temperature_therapist_gen} / TP {cfg.temperature_patient}, "
      f"max {cfg.max_tokens_per_response} tok)")
print(f"  Patient:      {cfg.patient_model_id}  (concurrency {cfg.patient_api_concurrency})")
print(f"  Batch size:   {cfg.conversation_batch_size}")
print(f"  Already on disk: {len(existing)} conversation CSV(s)"
      + ("  -> those are SKIPPED (per-CSV resume)" if existing else ""))
print("  No oracle calls, no branching, no look-ahead, no training.")

---
## 3. Seed verification — the pre-flight gate (free)

Replays the shuffle for every already-generated `model_iter_k` and checks each
conversation's patient really is the persona `seed + k + 1` predicts, by comparing the age
the patient states to the canonical persona's `age_value`.

Two decoy offsets are scored too. That matters: only a handful of distinct ages are spread
over the 96 personas, so *even a wrong shuffle* collides ~48% of the time — without the
decoys a pass would prove almost nothing. A decoy that does **not** fail is itself a failure.

Getting this wrong would not raise anywhere; it would silently break `persona_id` pairing,
and with it every paired statistic (`dz`, CIs) that joins this arm to another.

In [ ]:
# Therapist persona is drawn from the GLOBAL RNG inside setup_permutations — seed it first,
# exactly as train_PTO_Iterative.ipynb does, or the therapist prompt differs from iters 0..N-1.
random.seed(cfg.seed)
all_permutations, therapist_system_prompt, therapist_init_utterance = setup_permutations(
    only_expert_therapist=True,
)
print(f"Permutations: {len(all_permutations)}  |  therapist prompt: {len(therapist_system_prompt)} chars")

if VERIFY_SEEDS:
    if not verify_seeds(cfg, len(all_permutations)):
        raise SystemExit(
            "Seed verification did not pass — refusing to generate. Investigate before "
            "spending; a wrong shuffle silently breaks persona pairing."
        )

In [ ]:
# ─── The per-iteration persona order for THIS model state ────────────────────
shuffled = list(all_permutations)
random.Random(seed).shuffle(shuffled)
active_permutations = shuffled[: cfg.num_conversations_per_iter]
print(f"Active permutations: {len(active_permutations)} (shuffle seed {seed})")

---
## 4. Load the adapter

In [ ]:
client = init_openai_client(rt)
authenticate(rt, hf=True, wandb_enabled=False)   # HF token: Llama-3.2-1B is gated

tokenizer = setup_tokenizer(cfg.base_model_id)
# for_training=True matches the trainer notebook's load; run_generation_only flips
# use_cache back on and calls .eval(), so the generation path is identical.
base_policy = load_base_model(cfg.base_model_id, None, for_training=True)
sync_pad_token(base_policy, tokenizer)

policy = PeftModel.from_pretrained(base_policy, adapter_dir, is_trainable=False)
patch_generate(policy, tokenizer)   # re-patch: PeftModel wrapping drops the stop_strings bind
print(f"\u2713 Loaded adapter iteration_{ITER} onto {cfg.base_model_id} (bf16)")

---
## 5. Generate

Resume-safe per conversation CSV — an interruption costs only the in-flight batch, and
re-running this cell skips everything already saved.

Each batch prints `... batch <s>s, total <s>s, vram <N>G`. **Watch that `vram` number: it
should stay flat across batches.** It is `torch.cuda.memory_reserved()`. If it climbs every
batch you are accumulating allocator blocks and will eventually hit the card's limit — which
on the local Blackwell reboots the machine rather than raising `OutOfMemoryError`. Stop and
check that the inter-batch `empty_cache()` in `_shared/convs.py::_run_generation_rounds`
is still there.

In [ ]:
_states, gen_time, avg_len = run_generation_only(
    policy=policy, tokenizer=tokenizer, client=client,
    active_permutations=active_permutations,
    therapist_system_prompt=therapist_system_prompt,
    therapist_init_utterance=therapist_init_utterance,
    conv_dir=conv_dir, cfg=cfg,
    patient_api_seed=seed,
)
print(f"\n\u2713 {len(_states)} conversations in {gen_time / 60:.1f} min, avg len {avg_len:.1f}")

---
## 6. Confirm + next step

In [ ]:
final = sorted(f for f in os.listdir(conv_dir) if f.startswith("conversation_"))
print(f"{len(final)} / {cfg.num_conversations_per_iter} conversation CSVs in")
print(f"  {conv_dir}")

if len(final) < cfg.num_conversations_per_iter:
    print("\n  ! INCOMPLETE — re-run section 5; finished conversations are skipped.")
else:
    print("\n  \u2713 Complete. Next:")
    print("    1. Score with eda/notebooks/scoring/Run_Eval.ipynb — its EXPERIMENTS registry")
    print("       auto-discovers this arm from disk, so there is nothing to register.")
    print("    2. Second judge (Judge_Reliability.ipynb \u00a73) to keep the full-grid parity.")
    print("    3. python eda/tools/consolidate_scores.py build   (refresh the parquet fold)")
    print("    4. python eda/tools/render_results.py             (regenerate results/<top>/<sub>/)")